In [2]:
# ── UI パネル（パラメータ調整） ────────────────────

!pip install -q ipywidgets nest-asyncio edge-tts

from ipywidgets import IntSlider, Textarea, Dropdown, Button, VBox, HBox, Label, Output
from IPython.display import display, clear_output, Audio
import ipywidgets as widgets
import nest_asyncio
import asyncio
import datetime
import os
import edge_tts

nest_asyncio.apply()

LOCAL_AUDIO_DIR = "/content/audio_output"
os.makedirs(LOCAL_AUDIO_DIR, exist_ok=True)

VOICE_PRIORITY = [
    "ja-JP-NanamiNeural",
    "ja-JP-KeitaNeural",
]

# ─── 最後に生成されたファイルパスを保持するグローバル変数 ───
last_generated_audio = None

VOICE_PRIORITY = globals().get("VOICE_PRIORITY", VOICE_PRIORITY)

text_input = Textarea(
    value="こんにちは。今日はどんなお話をしましょうか？もしよければ、ゆっくりお話ししますね。",
    placeholder="読み上げるテキストを入力してください",
    description="テキスト:",
    rows=4,
    style={'description_width': '80px'}
)

voice_dropdown = Dropdown(
    options=VOICE_PRIORITY,
    value=VOICE_PRIORITY[0],
    description="音声:",
    style={'description_width': '80px'}
)

rate_slider = IntSlider(
    value=-5, min=-50, max=50, step=1,
    description="速度（Rate）:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
rate_label = Label(value="-5%")

pitch_slider = IntSlider(
    value=10, min=-50, max=50, step=1,
    description="ピッチ:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
pitch_label = Label(value="+10Hz")

volume_slider = IntSlider(
    value=0, min=-100, max=100, step=1,
    description="ボリューム:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
volume_label = Label(value="+0%")

def update_labels(change):
    rate_label.value = f"{rate_slider.value:+d}%"
    pitch_label.value = f"{pitch_slider.value:+d}Hz"
    volume_label.value = f"{volume_slider.value:+d}%"

rate_slider.observe(update_labels, 'value')
pitch_slider.observe(update_labels, 'value')
volume_slider.observe(update_labels, 'value')

execute_button = Button(
    description="🎵 実行（合成・再生）",
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

output_area = Output()

async def synthesize_with_ui():
    global last_generated_audio
    text = text_input.value
    voice = voice_dropdown.value
    rate_val = rate_slider.value
    pitch_val = pitch_slider.value
    volume_val = volume_slider.value

    output_area.clear_output()
    with output_area:
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"{LOCAL_AUDIO_DIR}/edge_{ts}.mp3"

        print(f"⏳ 処理中...")
        print(f"  テキスト: {text[:40]}...")
        print(f"  音声: {voice}")
        print(f"  Rate: {rate_val:+d}%, Pitch: {pitch_val:+d}Hz, Volume: {volume_val:+d}%")

        try:
            params = {}
            if rate_val != 0:
                params['rate'] = f"{rate_val:+d}%"
            if pitch_val != 0:
                params['pitch'] = f"{pitch_val:+d}Hz"
            if volume_val != 0:
                params['volume'] = f"{volume_val:+d}%"

            communicate = edge_tts.Communicate(text, voice=voice, **params)
            await communicate.save(output_path)

            # グローバル変数に保存（後工程セルで使用）
            last_generated_audio = output_path

            print(f"\n✅ 完了！")
            print(f"📁 保存: {output_path}")
            print(f"💡 セル2の後工程パネルで音声加工が可能です")
            print()
            display(Audio(output_path, autoplay=False))

        except Exception as e:
            print(f"\n❌ エラーが発生しました: {type(e).__name__}")
            print(f"   {e}")

def on_execute_clicked(button):
    asyncio.run(synthesize_with_ui())

execute_button.on_click(on_execute_clicked)

settings_box = VBox([
    Label(value="⚙️ 音声パラメータ調整", style={'font_weight': 'bold'}),
    HBox([rate_slider, rate_label]),
    HBox([pitch_slider, pitch_label]),
    HBox([volume_slider, volume_label]),
])

input_box = VBox([
    Label(value="📋 入力設定", style={'font_weight': 'bold'}),
    HBox([Label(value="テキスト:"), text_input]),
    HBox([Label(value="音声:"), voice_dropdown]),
])

control_panel = VBox([
    input_box,
    settings_box,
    execute_button,
    output_area
], layout=widgets.Layout(
    border='1px solid #ddd',
    padding='15px',
    border_radius='5px'
))

display(control_panel)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🎛️  セル2 ── 音声後工程パネル（声質・イントネーション加工）
#
# 使い方:
#   1. セル1で音声を合成する
#   2. このセルを実行してUIを表示する
#   3. プリセットまたは手動スライダーで加工パラメータを設定
#   4. 「🔊 加工して再生」ボタンを押す
# ══════════════════════════════════════════════════════════════

!pip install -q librosa soundfile pydub

import librosa
import librosa.effects
import soundfile as sf
import numpy as np
from pydub import AudioSegment
import ipywidgets as widgets
from ipywidgets import (
    FloatSlider, IntSlider, Dropdown, Button,
    VBox, HBox, Label, Output, HTML, ToggleButtons
)
from IPython.display import display, Audio
import datetime
import os

# ──────────────────────────────────────────────
# 定数
# ──────────────────────────────────────────────
LOCAL_AUDIO_DIR = "/content/audio_output"
os.makedirs(LOCAL_AUDIO_DIR, exist_ok=True)

# ══════════════════════════════════════════════
# 🎭 プリセット定義
# ── 声質ごとにピッチシフト量・速度・フォルマント補正を設定
# ══════════════════════════════════════════════
PRESETS = {
    "── プリセット選択 ──": {"pitch": 0.0,  "speed": 1.0,  "formant": 0.0,  "label": ""},
    "👦 子供（男の子）":     {"pitch": +5.0, "speed": 1.10, "formant": +1.5, "label": "child_boy"},
    "👧 子供（女の子）":     {"pitch": +7.0, "speed": 1.12, "formant": +2.0, "label": "child_girl"},
    "🧑 大人（男性）":       {"pitch": -3.0, "speed": 0.97, "formant": -1.0, "label": "adult_male"},
    "👩 大人（女性）":       {"pitch": +2.0, "speed": 1.00, "formant": +0.5, "label": "adult_female"},
    "🧓 老人（男性）":       {"pitch": -4.0, "speed": 0.88, "formant": -1.5, "label": "elder_male"},
    "👵 老人（女性）":       {"pitch": -1.0, "speed": 0.90, "formant": -0.5, "label": "elder_female"},
    "🤖 ロボット風":        {"pitch": 0.0,  "speed": 1.00, "formant": 0.0,  "label": "robot"},
    "✏️ 手動設定":          {"pitch": 0.0,  "speed": 1.0,  "formant": 0.0,  "label": "manual"},
}

# ══════════════════════════════════════════════
# 🎚️  スライダー群
# ══════════════════════════════════════════════

# --- ピッチシフト（半音単位）---
pitch_shift_slider = FloatSlider(
    value=0.0, min=-12.0, max=12.0, step=0.5,
    description="ピッチ(半音):",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px'),
    readout_format='.1f'
)
pitch_shift_label = Label(value="0.0 半音")

# --- 速度（タイムストレッチ）---
speed_slider = FloatSlider(
    value=1.0, min=0.5, max=2.0, step=0.05,
    description="速度(倍率):",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px'),
    readout_format='.2f'
)
speed_label = Label(value="×1.00")

# --- フォルマントシフト（擬似）---
formant_slider = FloatSlider(
    value=0.0, min=-3.0, max=3.0, step=0.25,
    description="声質シフト:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px'),
    readout_format='.2f'
)
formant_label = Label(value="0.00（中性）")

# --- イントネーション強調 ---
intonation_slider = FloatSlider(
    value=1.0, min=0.2, max=3.0, step=0.1,
    description="抑揚強調:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px'),
    readout_format='.1f'
)
intonation_label = Label(value="×1.0（標準）")

# --- ロボット化強度 ---
robot_slider = FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.05,
    description="ロボット強度:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px'),
    readout_format='.2f'
)
robot_label = Label(value="0.00（オフ）")

def update_slider_labels(change):
    pitch_shift_label.value  = f"{pitch_shift_slider.value:+.1f} 半音"
    speed_label.value         = f"×{speed_slider.value:.2f}"
    v = formant_slider.value
    tag = "低め" if v < -0.5 else ("高め" if v > 0.5 else "中性")
    formant_label.value       = f"{v:+.2f}（{tag}）"
    iv = intonation_slider.value
    itag = "フラット" if iv < 0.8 else ("強調" if iv > 1.5 else "標準")
    intonation_label.value    = f"×{iv:.1f}（{itag}）"
    robot_label.value         = f"{robot_slider.value:.2f}（{'オン' if robot_slider.value > 0 else 'オフ'}）"

for s in [pitch_shift_slider, speed_slider, formant_slider, intonation_slider, robot_slider]:
    s.observe(update_slider_labels, 'value')

# ══════════════════════════════════════════════
# 🎭 プリセット ドロップダウン
# ══════════════════════════════════════════════
preset_dropdown = Dropdown(
    options=list(PRESETS.keys()),
    value="── プリセット選択 ──",
    description="プリセット:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)

def on_preset_changed(change):
    """プリセット選択でスライダーを自動セット"""
    preset = PRESETS.get(change['new'], {})
    if preset.get('label') == 'manual':
        return  # 手動設定はスライダーをそのまま維持
    if preset.get('label') == 'robot':
        robot_slider.value = 0.8
        pitch_shift_slider.value = 0.0
        speed_slider.value = 1.0
        formant_slider.value = 0.0
        intonation_slider.value = 0.3
        return
    pitch_shift_slider.value  = preset.get('pitch', 0.0)
    speed_slider.value         = preset.get('speed', 1.0)
    formant_slider.value       = preset.get('formant', 0.0)
    intonation_slider.value    = 1.0
    robot_slider.value         = 0.0

preset_dropdown.observe(on_preset_changed, names='value')

# ══════════════════════════════════════════════
# 📂 入力ファイル選択
# ══════════════════════════════════════════════
source_dropdown = Dropdown(
    options=["セル1の最新ファイルを使用", "ファイルパスを直接指定"],
    value="セル1の最新ファイルを使用",
    description="入力元:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)

manual_path_input = widgets.Text(
    value="",
    placeholder="/content/audio_output/edge_XXXXXXXX.mp3",
    description="ファイルパス:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)
manual_path_box = VBox([manual_path_input],
    layout=widgets.Layout(display='none'))

def on_source_changed(change):
    if change['new'] == "ファイルパスを直接指定":
        manual_path_box.layout.display = ''
    else:
        manual_path_box.layout.display = 'none'

source_dropdown.observe(on_source_changed, names='value')

# ══════════════════════════════════════════════
# 🔧 音声加工コアロジック
# ══════════════════════════════════════════════

def mp3_to_wav(input_path: str) -> tuple:
    """MP3 → NumPy array (float32) + サンプルレート"""
    audio = AudioSegment.from_mp3(input_path)
    audio = audio.set_channels(1)           # モノラルに統一
    sr = audio.frame_rate
    samples = np.array(audio.get_array_of_samples(), dtype=np.float32)
    samples /= (2 ** (audio.sample_width * 8 - 1))  # −1〜+1 正規化
    return samples, sr

def apply_pitch_shift(y, sr, n_steps: float):
    """librosa でピッチシフト（半音単位）"""
    if abs(n_steps) < 0.01:
        return y
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def apply_time_stretch(y, rate: float):
    """librosa でタイムストレッチ（ピッチ変化なし）"""
    if abs(rate - 1.0) < 0.01:
        return y
    return librosa.effects.time_stretch(y, rate=rate)

def apply_formant_shift(y, sr, semitones: float):
    """
    擬似フォルマントシフト:
    ピッチを上下させた後に速度で元の長さに戻すことで
    フォルマント（共鳴特性）の移動を近似する
    """
    if abs(semitones) < 0.01:
        return y
    shift_ratio = 2 ** (semitones / 12.0)
    # ピッチを変えてから元の長さに戻す
    y_shifted = librosa.effects.pitch_shift(y, sr=sr, n_steps=semitones)
    y_restored = librosa.effects.time_stretch(y_shifted, rate=shift_ratio)
    # 元の長さに合わせる
    target_len = len(y)
    if len(y_restored) > target_len:
        return y_restored[:target_len]
    else:
        return np.pad(y_restored, (0, target_len - len(y_restored)))

def apply_intonation(y, sr, strength: float):
    """
    イントネーション強調:
    ピッチ輪郭を推定してスケールアップ/ダウンし、
    波形を PSOLA 的に並べ直す（近似的な実装）
    strength > 1 で抑揚が大きく、< 1 でフラットになる
    """
    if abs(strength - 1.0) < 0.05:
        return y
    # f0 推定
    f0, voiced_flag, _ = librosa.pyin(
        y, fmin=librosa.note_to_hz('C2'),
        fmax=librosa.note_to_hz('C7'), sr=sr
    )
    if f0 is None or np.all(np.isnan(f0)):
        return y

    # 有声区間のみ平均を計算
    voiced_f0 = f0[voiced_flag]
    if len(voiced_f0) == 0:
        return y
    mean_f0 = np.nanmean(voiced_f0)

    # フレームごとにピッチシフト量を計算して変換
    hop_length = 512
    frame_len = len(f0)
    output = y.copy()

    for i, (f, is_voiced) in enumerate(zip(f0, voiced_flag)):
        if not is_voiced or np.isnan(f) or f == 0:
            continue
        # 平均からの偏差を強調
        deviation_semitones = 12 * np.log2(f / mean_f0)
        target_semitones = deviation_semitones * strength
        delta = target_semitones - deviation_semitones

        # フレーム範囲を取得
        start = i * hop_length
        end   = min(start + hop_length * 2, len(y))
        chunk = y[start:end]
        if len(chunk) < 64:
            continue
        shifted = librosa.effects.pitch_shift(chunk, sr=sr, n_steps=delta)
        shifted_len = min(len(shifted), end - start)
        output[start:start + shifted_len] = shifted[:shifted_len]

    return output

def apply_robot_effect(y, sr, strength: float):
    """
    ロボット風エフェクト:
    一定間隔でピッチを固定する「ボコーダー近似」
    strength: 0=オフ, 1=最大
    """
    if strength < 0.01:
        return y
    # スペクトルの位相をランダム化して金属的質感を付与
    D = librosa.stft(y)
    magnitude = np.abs(D)
    # 位相をホワイトノイズに置換（強度に応じてブレンド）
    random_phase = np.exp(1j * np.random.uniform(0, 2 * np.pi, D.shape))
    robot_D = magnitude * random_phase
    y_robot = librosa.istft(robot_D, length=len(y))
    return (1 - strength) * y + strength * y_robot

# ══════════════════════════════════════════════
# 🔘 実行ボタン
# ══════════════════════════════════════════════
process_button = Button(
    description="🔊 加工して再生",
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)
proc_output = Output()

def on_process_clicked(btn):
    proc_output.clear_output()
    with proc_output:
        # ─ 入力ファイルの決定 ─
        if source_dropdown.value == "セル1の最新ファイルを使用":
            # グローバル変数から取得（セル1と同じカーネル）
            import builtins
            src = globals().get('last_generated_audio') or \
                  __builtins__.__dict__.get('last_generated_audio') if hasattr(__builtins__, '__dict__') \
                  else None
            # fallback: audio_output ディレクトリの最新ファイル
            if not src or not os.path.exists(src):
                files = sorted([
                    os.path.join(LOCAL_AUDIO_DIR, f)
                    for f in os.listdir(LOCAL_AUDIO_DIR)
                    if f.endswith('.mp3') and 'processed' not in f
                ])
                src = files[-1] if files else None
        else:
            src = manual_path_input.value.strip()

        if not src or not os.path.exists(src):
            print("❌ 入力ファイルが見つかりません。")
            print("  セル1で音声を生成するか、パスを直接指定してください。")
            return

        print(f"📂 入力: {src}")
        print("⏳ 加工中...")

        try:
            # 1. MP3 → NumPy
            y, sr = mp3_to_wav(src)
            print(f"  ✔ 読込完了 (sr={sr}, {len(y)/sr:.1f}秒)")

            # 2. ロボットエフェクト
            if robot_slider.value > 0:
                print(f"  🤖 ロボット加工 (強度={robot_slider.value:.2f})")
                y = apply_robot_effect(y, sr, robot_slider.value)

            # 3. 速度変更
            spd = speed_slider.value
            if abs(spd - 1.0) > 0.01:
                print(f"  ⏩ 速度調整 (×{spd:.2f})")
                y = apply_time_stretch(y, rate=spd)

            # 4. ピッチシフト
            ps = pitch_shift_slider.value
            if abs(ps) > 0.01:
                print(f"  🎵 ピッチシフト ({ps:+.1f}半音)")
                y = apply_pitch_shift(y, sr, ps)

            # 5. 擬似フォルマントシフト
            fs = formant_slider.value
            if abs(fs) > 0.01:
                print(f"  🗣️ フォルマントシフト ({fs:+.2f})")
                y = apply_formant_shift(y, sr, fs)

            # 6. イントネーション強調
            intn = intonation_slider.value
            if abs(intn - 1.0) > 0.05:
                print(f"  🎭 イントネーション強調 (×{intn:.1f})")
                y = apply_intonation(y, sr, intn)

            # 7. クリッピング防止
            max_val = np.max(np.abs(y))
            if max_val > 0:
                y = y / max_val * 0.95

            # 8. 保存
            ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            preset_tag = PRESETS.get(preset_dropdown.value, {}).get('label', 'custom')
            out_path = f"{LOCAL_AUDIO_DIR}/processed_{preset_tag}_{ts}.wav"
            sf.write(out_path, y, sr)

            print(f"\n✅ 加工完了！")
            print(f"📁 保存: {out_path}")
            print()
            display(Audio(out_path, autoplay=True))

        except Exception as e:
            import traceback
            print(f"\n❌ 加工エラー: {type(e).__name__}")
            print(f"   {e}")
            traceback.print_exc()

process_button.on_click(on_process_clicked)

# ══════════════════════════════════════════════
# 🎨 後工程パネル UI レイアウト
# ══════════════════════════════════════════════
header = HTML(value="""
<div style='
    background: linear-gradient(135deg, #1a1a2e, #16213e);
    color: #e0e0ff;
    padding: 12px 18px;
    border-radius: 6px 6px 0 0;
    font-family: monospace;
    font-size: 15px;
    letter-spacing: 1px;
'>
🎛️ &nbsp; 後工程パネル &mdash; 音声加工エンジン
</div>
""")

preset_box = VBox([
    Label(value="🎭 声質プリセット", style={'font_weight': 'bold'}),
    preset_dropdown,
])

slider_box = VBox([
    Label(value="🎚️ 手動調整", style={'font_weight': 'bold'}),
    HBox([pitch_shift_slider,  pitch_shift_label]),
    HBox([speed_slider,        speed_label]),
    HBox([formant_slider,      formant_label]),
    HBox([intonation_slider,   intonation_label]),
    HBox([robot_slider,        robot_label]),
])

source_box = VBox([
    Label(value="📂 入力ファイル", style={'font_weight': 'bold'}),
    source_dropdown,
    manual_path_box,
])

post_panel = VBox([
    header,
    VBox([
        source_box,
        preset_box,
        slider_box,
        process_button,
        proc_output,
    ], layout=widgets.Layout(padding='15px'))
], layout=widgets.Layout(
    border='2px solid #1a1a2e',
    border_radius='8px',
    margin='10px 0'
))

display(post_panel)

c:\work\HP4\deepxde\.venv\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
